# SIKAFE — M2: entrenar y evaluar YOLO con fotos reales de café (Croppie)

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU*** y guardar.

Este notebook:
1. Descarga el dataset **Croppie** (CC BY-SA 4.0, Producers Direct y Alliance Bioversity & CIAT).
2. Lo prepara: 4 clases SIKAFE, partición 70/15/15 **por foto original** (sin fuga), estadísticas.
3. Entrena **YOLO11n** (transfer learning desde COCO) y guarda todo en Google Drive (`MyDrive/sikafe/`).
4. Evalúa con las métricas del proyecto, a través del mismo adaptador que usa la app.

Ejecute las celdas **en orden**. Al final, **copie y pegue en el chat** la salida de las celdas marcadas con 📋.
Si Colab se desconecta durante el entrenamiento, vea la sección *Reanudar*.

In [ ]:
# 1. Comprobar la GPU (debe aparecer una Tesla T4 o similar)
!nvidia-smi

In [ ]:
# 2. Conectar Google Drive (aquí se guardan las corridas: sobreviven a desconexiones)
from google.colab import drive
drive.mount('/content/drive')
SALIDA = '/content/drive/MyDrive/sikafe'
!mkdir -p {SALIDA}/corridas {SALIDA}/datos

In [ ]:
# 3. Traer el código del proyecto (la rama debe estar subida a GitHub)
# salir de la carpeta antes de borrarla (si no, git clone falla al re-ejecutar)
%cd /content
RAMA = 'fase-2-3-croppie-yolo'
!rm -rf /content/rendimientocosecha
!git clone -q -b {RAMA} https://github.com/Darwin-md11/rendimientocosecha /content/rendimientocosecha
%cd /content/rendimientocosecha
!git log --oneline -1

In [ ]:
# 4. Dependencias del laboratorio (Colab ya trae PyTorch con CUDA)
!pip install -q -r requirements-ml.txt
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
# 5. Descargar Croppie desde Hugging Face (~931 MB)
import os, zipfile
from huggingface_hub import hf_hub_download
RAW = '/content/croppie_raw'
for archivo in ['data/train.zip', 'data/val.zip']:
    ruta = hf_hub_download('rgautroncgiar/croppie_coffee_ug', archivo, repo_type='dataset')
    destino = os.path.join(RAW, os.path.basename(archivo)[:-4])
    zipfile.ZipFile(ruta).extractall(destino)
    print('extraído', archivo, '->', destino)
!find {RAW} -maxdepth 3 -type d | head -20

### 📋 6. Inspeccionar nombres y agrupamiento
Muestra cuántas imágenes hay, cómo se llaman y cómo se agrupan los recortes por foto original.
Si `patron_elegido` sale **NINGUNO**, pegue esta salida en el chat antes de seguir.

In [ ]:
!python -m ml.datos.preparar_yolo --origen {RAW} --inspeccionar

### 📋 7. Preparar el dataset (determinista: misma semilla → mismo dataset)
Genera `data.yaml`, el manifiesto con hashes, la prueba de fuga y el análisis exploratorio (tabla + gráfico para la sección 8.2).

In [ ]:
DATOS = '/content/sikafe_yolo'
!python -m ml.datos.preparar_yolo --origen {RAW} --destino {DATOS} --dataset croppie --semilla 0
!cp {DATOS}/eda.md {DATOS}/eda_clases.png {DATOS}/estadisticas.json {DATOS}/manifiesto.json {SALIDA}/datos/

### 8. Entrenar YOLO11n a 640 px
Puede tardar bastante (depende de la GPU asignada y de la parada temprana). Todo queda en `MyDrive/sikafe/corridas/<id>`.

In [ ]:
!python -m ml.m2_yolo.entrenar --config ml/experiments/configs/m2_yolo11n_640.yaml --datos {DATOS}/data.yaml --salida {SALIDA}/corridas

### Reanudar (solo si Colab se desconectó durante el paso 8)
Vuelva a ejecutar las celdas 1 a 5 y la 7 (el dataset sale idéntico), y luego esta:

In [ ]:
import glob
CORRIDA = sorted(glob.glob(f'{SALIDA}/corridas/m2_yolo11n_640_*'))[-1]
print('Reanudando', CORRIDA)
!python -m ml.m2_yolo.entrenar --reanudar {CORRIDA}

### 📋 9. Evaluar en validación (se puede repetir)

In [ ]:
import glob
CORRIDA = sorted(glob.glob(f'{SALIDA}/corridas/m2_yolo11n_640_*'))[-1]
print('Corrida:', CORRIDA)
!python -m ml.m2_yolo.evaluar --corrida {CORRIDA} --split val

In [ ]:
# 📋 9b. Resumen de la evaluación y curva de entrenamiento (últimas épocas)
!cat {CORRIDA}/resumen_val.md
!tail -n 10 {CORRIDA}/train/results.csv

### 📋 10. Contrato del detector con el modelo real
Comprueba con los pesos reales lo mismo que la app exige a todo detector (cajas dentro de la imagen, umbral, determinismo, imágenes límite).

In [ ]:
import sys, glob
from PIL import Image
sys.path.insert(0, 'tests')
from contrato_detector import verificar_contrato
from src.infrastructure.adapters.detectors.yolo_detector_adapter import YoloFruitDetectorAdapter
det = YoloFruitDetectorAdapter.desde_archivo(f'{CORRIDA}/m2_yolo.pt')
img = Image.open(sorted(glob.glob(f'{DATOS}/images/val/*'))[0]).convert('RGB')
verificar_contrato(det, img)
dets = det.detectar_frutos(img, 0.25)
print('Contrato OK |', len(dets), 'cerezas en', img.size, '|', det.info())

### 📋 M0 — línea base clásica (color + textura + RF/SVM sobre recortes)
Corre en **CPU** (no necesita GPU). Usa los recortes de las cajas anotadas de la **misma partición** que M2.
La primera vez extrae las características de todos los recortes (varios minutos) y las guarda en caché; SVM las reutiliza.
El tiempo de extracción queda en `tiempos.json`: es la línea base si se acelera este paso con CUDA o MPI.


In [ ]:
!python -m ml.m0_clasico.entrenar --datos {DATOS}/data.yaml --salida {SALIDA}/corridas --modelo rf


In [ ]:
!python -m ml.m0_clasico.entrenar --datos {DATOS}/data.yaml --salida {SALIDA}/corridas --modelo svm


### 📋 M1 — CNN con transfer learning (GPU)
MobileNetV2 preentrenada en ImageNet, ajustada para clasificar la madurez de los **mismos recortes** que M0 (misma partición).
Primero la **prueba rápida** (~2 min, pocos recortes): si falla, pegue el error en el chat antes de lanzar la corrida completa.
La primera vez se crea la caché de recortes en Drive (unos minutos); después se reutiliza.

In [ ]:
!python -m ml.m1_cnn.entrenar --config ml/experiments/configs/m1_mobilenetv2_96.yaml --datos {DATOS}/data.yaml --salida {SALIDA}/corridas --rapido

In [ ]:
!python -m ml.m1_cnn.entrenar --config ml/experiments/configs/m1_mobilenetv2_96.yaml --datos {DATOS}/data.yaml --salida {SALIDA}/corridas

In [ ]:
# (Opcional, si hay tiempo) Experimento (d): la misma red SIN preentrenar, y otra arquitectura
# !python -m ml.m1_cnn.entrenar --config ml/experiments/configs/m1_mobilenetv2_96_sin_preentrenar.yaml --datos {DATOS}/data.yaml --salida {SALIDA}/corridas
# !python -m ml.m1_cnn.entrenar --config ml/experiments/configs/m1_efficientnet_b0_96.yaml --datos {DATOS}/data.yaml --salida {SALIDA}/corridas

### 📋 CUDA — aceleración en GPU (tabla para la sección 8.4)
Requiere **T4 GPU**. (1) El descriptor de M0 como **kernel CUDA propio** (Numba, un hilo por recorte), comparado con la CPU
sobre 20 000 recortes reales, **verificando que den el mismo resultado**; (2) la inferencia de YOLO en CPU frente a GPU.
Si la primera celda termina con `ERROR`, pegue la salida en el chat.

In [ ]:
!python -m ml.benchmark.cuda_m0 --datos {DATOS}/data.yaml --split val --n 20000 --salida {SALIDA}/benchmarks

In [ ]:
!python -m ml.benchmark.yolo_cpu_gpu --corrida {CORRIDA} --n 200 --salida {SALIDA}/benchmarks

### 📋 11. Evaluación FINAL en test (una sola vez)
Ejecútela **solo cuando el equipo haya decidido que este es el modelo** (después de mirar validación).
El protocolo exige tocar el test una única vez; repetirla queda registrado.

In [ ]:
!python -m ml.m2_yolo.evaluar --corrida {CORRIDA} --split test --final

In [ ]:
# 11b. Evaluación FINAL en test de M0 y M1 (una sola vez por modelo) — desactivada
import glob, subprocess, sys
EJECUTAR_TEST_FINAL_M0_M1 = False
def ultima(patron):
    rutas = [r for r in sorted(glob.glob(f'{SALIDA}/corridas/{patron}')) if '_rapido' not in r]
    return rutas[-1] if rutas else None
if EJECUTAR_TEST_FINAL_M0_M1:
    for modulo, patron in [('ml.m0_clasico.entrenar', 'm0_rf_*'), ('ml.m0_clasico.entrenar', 'm0_svm_*'), ('ml.m1_cnn.entrenar', 'm1_mobilenetv2_96_s[0-9]*'), ('ml.m1_cnn.entrenar', 'm1_mobilenetv2_96_sin_preentrenar_s*')]:
        corrida = ultima(patron)
        print('==>', modulo, corrida)
        if corrida:
            print(subprocess.run([sys.executable, '-m', modulo, '--corrida', corrida, '--final'], capture_output=True, text=True).stdout[-3000:])
else:
    print('Test final de M0/M1 NO ejecutado (protocolo: una sola vez, cuando se decidan los modelos).')


In [ ]:
# 📋 12. Todo lo que hay que pegar en el chat
!cat {CORRIDA}/entorno.json
!cat {CORRIDA}/metricas_val_ultralytics.json
!cat {CORRIDA}/m2_yolo.manifest.json
!ls -la {CORRIDA}

### 📋 14. Tablas y figuras del capítulo 8 (8.2–8.4)
Lee las cinco corridas **oficiales** de `configs/informe.yaml` (ID exactos, sin patrones) y genera `informe_resultados.md`, `tablas.json` y `figuras/` en `MyDrive/sikafe/informe`. No entrena ni toca el test de nuevo: recalcula las métricas de M2 desde las predicciones guardadas para sacar intervalos de confianza (bootstrap, ~2–5 min).
Necesita el dataset preparado (celda 7) para leer las etiquetas de test.

In [ ]:
!python -m ml.informe.generar --corridas {SALIDA}/corridas --datos {DATOS}/data.yaml --benchmarks {SALIDA}/benchmarks --salida {SALIDA}/informe

### 13. Llevar el modelo a la app
Descargue de `MyDrive/sikafe/corridas/<id>/` los archivos **`m2_yolo.pt`** y **`m2_yolo.manifest.json`** y cópielos a la carpeta `models/` del repositorio.
Con eso la app usa YOLO por defecto (`SIKAFE_DETECTOR=yolo`). Requiere `pip install -r requirements-ml.txt` en ese computador.

### (Opcional) Experimento (c): 960 px
Mismo flujo con `ml/experiments/configs/m2_yolo11n_960.yaml`. Ojo: los recortes de Croppie miden 480×640, así que 960 px **amplía** la imagen; el efecto esperado es sobre cerezas pequeñas.

In [ ]:
# !python -m ml.m2_yolo.entrenar --config ml/experiments/configs/m2_yolo11n_960.yaml --datos {DATOS}/data.yaml --salida {SALIDA}/corridas